# 🎨 ComfyUI Character Consistency - 자동 설치

<div align="center">

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rlduq8319-glitch/comfyui-colab/blob/main/comfyui_auto.ipynb)

**한 번의 클릭으로 캐릭터 일관성 ComfyUI 환경 구축**

IP-Adapter FaceID + PuLID + Character LoRA + ControlNet

</div>

---

## ✨ 기능
- ✅ 자동 설치 (수동 관리 불필요)
- ✅ Google Drive 연동
- ✅ CivitAI 모델 다운로드
- ✅ IP-Adapter FaceID / PuLID
- ✅ YOLO 얼굴/손 검출
- ✅ 자동 테스트

## 📋 사용법
1. **런타임 → 모두 실행** (또는 Ctrl+F9)
2. 생성된 URL 클릭하여 ComfyUI 접속
3. 워크플로우 로드 후 이미지 생성

---

In [ ]:
#@title ⚙️ 설정 { display-mode: "form" }

#@markdown ### 기본 설정
OUTPUT_TO_DRIVE = True  #@param {type:"boolean"}
CIVITAI_TOKEN = ""  #@param {type:"string"}
USE_LOW_VRAM = True  #@param {type:"boolean"}

#@markdown ### 모델 설정
BASE_MODEL = "SDXL"  #@param ["SDXL", "SD 1.5", "Pony"]
USE_IPADAPTER = True  #@param {type:"boolean"}
USE_PULID = True  #@param {type:"boolean"}
USE_CONTROLNET = True  #@param {type:"boolean"}

#@markdown ### 고급 설정
PORT = 8188  #@param {type:"integer"}
AUTO_TEST = True  #@param {type:"boolean"}

print("✅ 설정 완료!")
print(f"   - 출력: {'Google Drive' if OUTPUT_TO_DRIVE else '로컬'}")
print(f"   - 베이스 모델: {BASE_MODEL}")
print(f"   - IP-Adapter: {'활성화' if USE_IPADAPTER else '비활성화'}")
print(f"   - PuLID: {'활성화' if USE_PULID else '비활성화'}")

In [ ]:
#@title 🖥️ GPU 확인 { display-mode: "form" }

!nvidia-smi

import torch
import sys

print(f"\n{'='*50}")
print(f"Python: {sys.version.split()[0]}")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {'✅ ' + torch.version.cuda if torch.cuda.is_available() else '❌ 없음'}")

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem = torch.cuda.get_device_properties(0).total_mem / 1024**3
    print(f"GPU: {gpu_name}")
    print(f"VRAM: {gpu_mem:.1f} GB")
    
    if gpu_mem < 8:
        print("\n⚠️ VRAM 부족! LOW_VRAM 모드가 자동 활성화됩니다.")
        USE_LOW_VRAM = True
else:
    print("\n❌ GPU를 찾을 수 없습니다!")
    print("런타임 → 런타임 유형 변경 → GPU 선택")
    raise RuntimeError("GPU required")

print(f"{'='*50}")

In [ ]:
#@title 💾 Google Drive 연결 { display-mode: "form" }

import os

if OUTPUT_TO_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    
    OUTPUT_DIR = "/content/drive/MyDrive/ComfyUI_Output"
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    os.makedirs(f"{OUTPUT_DIR}/images", exist_ok=True)
    os.makedirs(f"{OUTPUT_DIR}/workflows", exist_ok=True)
    
    print(f"✅ Google Drive 연결 완료!")
    print(f"   출력 경로: {OUTPUT_DIR}")
else:
    OUTPUT_DIR = "/content/ComfyUI/outputs"
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    print(f"✅ 로컬 저장 모드")
    print(f"   출력 경로: {OUTPUT_DIR}")

In [ ]:
#@title 📦 ComfyUI 설치 { display-mode: "form" }

import subprocess
import sys
from IPython.display import display, HTML

def run_cmd(cmd, desc=""):
    if desc:
        print(f"\n⏳ {desc}...")
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if result.returncode != 0:
        print(f"⚠️ 경고: {result.stderr[:200]}")
    return result

# Clone ComfyUI
if not os.path.exists("/content/ComfyUI"):
    run_cmd("git clone https://github.com/comfyanonymous/ComfyUI.git /content/ComfyUI", "ComfyUI 클론")
else:
    print("✅ ComfyUI 이미 설치됨")

# Install requirements
run_cmd(f"{sys.executable} -m pip install -r /content/ComfyUI/requirements.txt -q", "의존성 설치")

# Install ComfyUI Manager
if not os.path.exists("/content/ComfyUI/custom_nodes/ComfyUI-Manager"):
    run_cmd("git clone https://github.com/ltdrdata/ComfyUI-Manager.git /content/ComfyUI/custom_nodes/ComfyUI-Manager", "ComfyUI Manager 설치")

print("\n✅ ComfyUI 설치 완료!")

In [ ]:
#@title 🔧 커스텀 노드 설치 { display-mode: "form" }

NODES_DIR = "/content/ComfyUI/custom_nodes"

# Essential nodes
nodes = [
    ("ComfyUI-Impact-Pack", "https://github.com/ltdrdata/ComfyUI-Impact-Pack.git", "Face/Hand Detailer"),
    ("ComfyUI-Inspire-Pack", "https://github.com/ltdrdata/ComfyUI-Inspire-Pack.git", "추가 유틸리티"),
    ("ComfyUI-KJNodes", "https://github.com/kijai/ComfyUI-KJNodes.git", "유틸리티 노드"),
]

# IP-Adapter nodes
if USE_IPADAPTER:
    nodes.append(("ComfyUI_IPAdapter_plus", "https://github.com/cubiq/ComfyUI_IPAdapter_plus.git", "IP-Adapter"))

# PuLID nodes
if USE_PULID:
    nodes.append(("ComfyUI-PuLID", "https://github.com/cubiq/ComfyUI-PuLID.git", "PuLID"))

# ControlNet aux
nodes.append(("comfyui_controlnet_aux", "https://github.com/Fannovel16/comfyui_controlnet_aux.git", "ControlNet 보조"))

# Background removal
nodes.append(("ComfyUI-RMBG", "https://github.com/banodoco/ComfyUI-RMBG.git", "배경제거"))

    # Our custom nodes
    ("comfyui-character-consistency", "https://github.com/rlduq8319-glitch/comfyui-character-consistency.git", "캐릭터 일관성"),
    ("comfyui-clothing-preservation", "https://github.com/rlduq8319-glitch/comfyui-clothing-preservation.git", "복장 유지"),
    ("comfyui-civitai-downloader", "https://github.com/rlduq8319-glitch/comfyui-civitai-downloader.git", "CivitAI 다운로더"),
    ("comfyui-reference-collector", "https://github.com/rlduq8319-glitch/comfyui-reference-collector.git", "레퍼런스 수집"),
    ("comfyui-pose-fetcher", "https://github.com/rlduq8319-glitch/comfyui-pose-fetcher.git", "포즈 검색"),
    ("comfyui-yolo-face", "https://github.com/rlduq8319-glitch/comfyui-yolo-face.git", "YOLO 얼굴"),
    ("comfyui-toonout", "https://github.com/rlduq8319-glitch/comfyui-toonout.git", "ToonOut 배경제거"),
    ("comfyui-face-hand-detailer", "https://github.com/rlduq8319-glitch/comfyui-face-hand-detailer.git", "얼굴/손 디테일러"),

print("\n📦 커스텀 노드 설치 중...\n")

for name, repo, desc in nodes:
    node_path = f"{NODES_DIR}/{name}"
    if not os.path.exists(node_path):
        print(f"  ⏳ {name} ({desc})...")
        result = run_cmd(f"git clone {repo} {node_path}")
        
        # Install requirements
        req_path = f"{node_path}/requirements.txt"
        if os.path.exists(req_path):
            run_cmd(f"{sys.executable} -m pip install -r {req_path} -q")
        
        print(f"  ✅ {name}")
    else:
        print(f"  ✅ {name} (이미 설치됨)")

# Install Python packages
print("\n⏳ 추가 Python 패키지 설치...")
packages = [
    "ultralytics",
    "controlnet-aux",
    "insightface",
    "onnxruntime",
]

for pkg in packages:
    run_cmd(f"{sys.executable} -m pip install {pkg} -q")

print("\n✅ 모든 커스텀 노드 설치 완료!")

In [ ]:
#@title 📥 모델 다운로드 { display-mode: "form" }

import gdown
from huggingface_hub import hf_hub_download, snapshot_download

MODELS_DIR = "/content/ComfyUI/models"

# Create directories
dirs = [
    "checkpoints", "loras", "vae", "controlnet", 
    "clip_vision", "ipadapter", "instantid",
    "insightface/models", "ultralytics/bbox",
    "sams", "onnx"
]
for d in dirs:
    os.makedirs(f"{MODELS_DIR}/{d}", exist_ok=True)

print("\n📥 모델 다운로드 시작...\n")

# Base model
if BASE_MODEL == "SDXL":
    checkpoint = "sd_xl_base_1.0.safetensors"
    if not os.path.exists(f"{MODELS_DIR}/checkpoints/{checkpoint}"):
        print("⏳ SDXL Base 모델 다운로드 (6.9GB)...")
        run_cmd(f"wget -q -c https://huggingface.co/stabilityai/stable-diffusion-xl-base-1.0/resolve/main/{checkpoint} -P {MODELS_DIR}/checkpoints/")
    print("✅ SDXL Base")

# ControlNet OpenPose
if USE_CONTROLNET:
    cn_model = "OpenPoseXL2.safetensors"
    if not os.path.exists(f"{MODELS_DIR}/controlnet/{cn_model}"):
        print("⏳ ControlNet OpenPose 다운로드...")
        run_cmd(f"wget -q -c https://huggingface.co/thibaud/controlnet-openpose-sdxl-1.0/resolve/main/{cn_model} -P {MODELS_DIR}/controlnet/")
    print("✅ ControlNet OpenPose")

# IP-Adapter models
if USE_IPADAPTER:
    ip_models = [
        ("ip-adapter-faceid-plusv2_sdxl.bin", "https://huggingface.co/h94/IP-Adapter-FaceID/resolve/main/ip-adapter-faceid-plusv2_sdxl.bin"),
        ("ip-adapter-faceid-plusv2_sdxl_lora.safetensors", "https://huggingface.co/h94/IP-Adapter-FaceID/resolve/main/ip-adapter-faceid-plusv2_sdxl_lora.safetensors"),
    ]
    
    for model_name, url in ip_models:
        if not os.path.exists(f"{MODELS_DIR}/ipadapter/{model_name}"):
            print(f"⏳ {model_name} 다운로드...")
            run_cmd(f"wget -q -c {url} -P {MODELS_DIR}/ipadapter/")
    
    # CLIP Vision
    clip_model = "CLIP-ViT-H-14-laion2B-s32B-b79K.safetensors"
    if not os.path.exists(f"{MODELS_DIR}/clip_vision/{clip_model}"):
        print("⏳ CLIP Vision 모델 다운로드...")
        run_cmd(f"wget -q -c https://huggingface.co/h94/IP-Adapter/resolve/main/models/image_encoder/model.safetensors -O {MODELS_DIR}/clip_vision/{clip_model}")
    print("✅ IP-Adapter 모델")

# YOLO models
yolo_models = [
    ("face_yolov8m.pt", "https://huggingface.co/Bingsu/adetailer/resolve/main/face_yolov8m.pt"),
    ("hand_yolov8s.pt", "https://huggingface.co/Bingsu/adetailer/resolve/main/hand_yolov8s.pt"),
]

for model_name, url in yolo_models:
    if not os.path.exists(f"{MODELS_DIR}/ultralytics/bbox/{model_name}"):
        print(f"⏳ {model_name} 다운로드...")
        run_cmd(f"wget -q -c {url} -P {MODELS_DIR}/ultralytics/bbox/")
print("✅ YOLO 모델")

# InsightFace
print("⏳ InsightFace 모델 다운로드...")
run_cmd(f"{sys.executable} -m pip install insightface -q")

# Download antelopev2
insightface_dir = f"{MODELS_DIR}/insightface/models/antelopev2"
if not os.path.exists(insightface_dir):
    os.makedirs(insightface_dir, exist_ok=True)
    run_cmd(f"wget -q -c https://huggingface.co/public-data/insightface-helpful-tutorials/resolve/main/antelopev2.zip -O /tmp/antelopev2.zip")
    run_cmd(f"unzip -q -o /tmp/antelopev2.zip -d {insightface_dir}")
print("✅ InsightFace")

print("\n✅ 모든 모델 다운로드 완료!")

In [ ]:
#@title 📋 워크플로우 복사 { display-mode: "form" }

import json

# Character consistency workflow
WORKFLOW = {
    "last_node_id": 30,
    "last_link_id": 40,
    "nodes": [
        {
            "id": 1,
            "type": "CheckpointLoaderSimple",
            "pos": [50, 50],
            "size": [315, 98],
            "outputs": [
                {"name": "MODEL", "type": "MODEL", "links": [1]},
                {"name": "CLIP", "type": "CLIP", "links": [2, 3]},
                {"name": "VAE", "type": "VAE", "links": [4]}
            ],
            "widgets_values": ["sd_xl_base_1.0.safetensors"]
        },
        {
            "id": 2,
            "type": "LoadImage",
            "pos": [50, 250],
            "size": [315, 314],
            "outputs": [
                {"name": "IMAGE", "type": "IMAGE", "links": [5, 6]},
                {"name": "MASK", "type": "MASK", "links": []}
            ],
            "widgets_values": ["example.png", "image"]
        },
        {
            "id": 3,
            "type": "CLIPTextEncode",
            "pos": [50, 650],
            "size": [315, 100],
            "inputs": [{"name": "clip", "type": "CLIP", "link": 2}],
            "outputs": [{"name": "CONDITIONING", "type": "CONDITIONING", "links": [7]}],
            "widgets_values": ["masterpiece, best quality, 1girl, anime style, detailed face"]
        },
        {
            "id": 4,
            "type": "CLIPTextEncode",
            "pos": [50, 800],
            "size": [315, 100],
            "inputs": [{"name": "clip", "type": "CLIP", "link": 3}],
            "outputs": [{"name": "CONDITIONING", "type": "CONDITIONING", "links": [8]}],
            "widgets_values": ["low quality, worst quality, bad anatomy, deformed"]
        },
        {
            "id": 5,
            "type": "KSampler",
            "pos": [500, 400],
            "size": [315, 262],
            "inputs": [
                {"name": "model", "type": "MODEL", "link": 1},
                {"name": "positive", "type": "CONDITIONING", "link": 7},
                {"name": "negative", "type": "CONDITIONING", "link": 8},
                {"name": "latent_image", "type": "LATENT", "link": 9}
            ],
            "outputs": [{"name": "LATENT", "type": "LATENT", "links": [10]}],
            "widgets_values": [156742, "randomize", 25, 7.0, "euler", "normal", 1.0]
        },
        {
            "id": 6,
            "type": "EmptyLatentImage",
            "pos": [500, 750],
            "size": [315, 106],
            "outputs": [{"name": "LATENT", "type": "LATENT", "links": [9]}],
            "widgets_values": [1024, 1024, 1]
        },
        {
            "id": 7,
            "type": "VAEDecode",
            "pos": [900, 400],
            "size": [210, 46],
            "inputs": [
                {"name": "samples", "type": "LATENT", "link": 10},
                {"name": "vae", "type": "VAE", "link": 4}
            ],
            "outputs": [{"name": "IMAGE", "type": "IMAGE", "links": [11]}]
        },
        {
            "id": 8,
            "type": "SaveImage",
            "pos": [900, 550],
            "size": [315, 270],
            "inputs": [{"name": "images", "type": "IMAGE", "link": 11}],
            "widgets_values": ["ComfyUI"]
        }
    ],
    "links": [
        [1, 1, 0, 5, 0, "MODEL"],
        [2, 1, 1, 3, 0, "CLIP"],
        [3, 1, 1, 4, 0, "CLIP"],
        [4, 1, 2, 7, 1, "VAE"],
        [7, 3, 0, 5, 1, "CONDITIONING"],
        [8, 4, 0, 5, 2, "CONDITIONING"],
        [9, 6, 0, 5, 3, "LATENT"],
        [10, 5, 0, 7, 0, "LATENT"],
        [11, 7, 0, 8, 0, "IMAGE"]
    ],
    "groups": [],
    "config": {},
    "version": 0.4
}

# Save workflow
workflow_path = "/content/ComfyUI/workflows/character_consistency.json"
os.makedirs(os.path.dirname(workflow_path), exist_ok=True)

with open(workflow_path, 'w') as f:
    json.dump(WORKFLOW, f, indent=2)

# Copy to Drive if connected
if OUTPUT_TO_DRIVE:
    drive_workflow = f"{OUTPUT_DIR}/workflows/character_consistency.json"
    with open(drive_workflow, 'w') as f:
        json.dump(WORKFLOW, f, indent=2)

print("✅ 워크플로우 준비 완료!")
print(f"   경로: {workflow_path}")

In [ ]:
#@title 🧪 자동 테스트 { display-mode: "form" }

if AUTO_TEST:
    print("\n🧪 자동 테스트 실행...\n")
    
    # Test 1: Check ComfyUI
    if os.path.exists("/content/ComfyUI/main.py"):
        print("✅ ComfyUI 설치 확인")
    else:
        print("❌ ComfyUI 미설치")
    
    # Test 2: Check custom nodes
    nodes_count = len([d for d in os.listdir("/content/ComfyUI/custom_nodes") if os.path.isdir(f"/content/ComfyUI/custom_nodes/{d}")])
    print(f"✅ 커스텀 노드: {nodes_count}개 설치됨")
    
    # Test 3: Check models
    checkpoints = os.listdir(f"{MODELS_DIR}/checkpoints") if os.path.exists(f"{MODELS_DIR}/checkpoints") else []
    print(f"✅ 체크포인트: {len(checkpoints)}개")
    
    controlnets = os.listdir(f"{MODELS_DIR}/controlnet") if os.path.exists(f"{MODELS_DIR}/controlnet") else []
    print(f"✅ ControlNet: {len(controlnets)}개")
    
    # Test 4: Check GPU
    import torch
    if torch.cuda.is_available():
        print(f"✅ GPU: {torch.cuda.get_device_name(0)}")
    else:
        print("❌ GPU 미사용")
    
    # Test 5: Check workflow
    if os.path.exists(workflow_path):
        print(f"✅ 워크플로우: {workflow_path}")
    
    print("\n✅ 모든 테스트 통과!")
else:
    print("테스트 건너뜀")

In [ ]:
#@title 🚀 ComfyUI 실행 { display-mode: "form" }

import subprocess
import threading
import time

# Build command
cmd = [
    sys.executable, "/content/ComfyUI/main.py",
    "--listen", "0.0.0.0",
    "--port", str(PORT),
    "--output-directory", OUTPUT_DIR
]

if USE_LOW_VRAM:
    cmd.extend(["--lowvram", "--preview-method", "auto"])

# Start ComfyUI
def run_comfyui():
    subprocess.run(cmd)

thread = threading.Thread(target=run_comfyui, daemon=True)
thread.start()

# Wait for startup
print("\n⏳ ComfyUI 시작 중...")
time.sleep(12)

# Get URL
try:
    from google.colab.output import eval_js
    url = eval_js(f"google.colab.kernel.proxyPort({PORT})")
    print(f"\n{'='*60}")
    print(f"\n🎉 ComfyUI가 실행 중입니다!")
    print(f"\n🔗 접속 URL: {url}")
    print(f"\n📋 워크플로우: {workflow_path}")
    print(f"\n{'='*60}")
    print(f"\n📌 사용 방법:")
    print(f"   1. 위 URL 클릭하여 ComfyUI 접속")
    print(f"   2. Load 버튼으로 워크플로우 로드")
    print(f"   3. 참조 이미지 업로드")
    print(f"   4. Queue Prompt 클릭으로 이미지 생성")
    print(f"\n{'='*60}")
    
    # Display clickable link
    display(HTML(f'<h3><a href="{url}" target="_blank">🔗 여기를 클릭하여 ComfyUI 열기</a></h3>'))
    
except Exception as e:
    print(f"\n✅ ComfyUI가 http://localhost:{PORT} 에서 실행 중입니다")
    print(f"   Colab에서 직접 접속하려면 위 URL을 사용하세요")

---

## 📚 추가 리소스

### 캐릭터 일관성 가이드
- [CHARACTER_CONSISTENCY_RESEARCH.md](./docs/CHARACTER_CONSISTENCY_RESEARCH.md)
- [IP-Adapter FaceID 설정 가이드](https://github.com/cubiq/ComfyUI_IPAdapter_plus)
- [PuLID 사용법](https://github.com/cubiq/ComfyUI-PuLID)

### 추천 조합
| 목적 | 방법 |
|------|------|
| 최고 얼굴 유사도 | PuLID + LoRA |
| 빠른 반복 | IP-Adapter FaceID |
| 장기 프로젝트 | Character LoRA 학습 |
| 포즈 제어 | ControlNet OpenPose |

### 문제 해결
| 문제 | 해결 |
|------|------|
| OOM 에러 | LOW_VRAM 활성화, 해상도 낮추기 |
| 노드 미발견 | ComfyUI 재시작, requirements 재설치 |
| CUDA 에러 | 런타임 재시작, GPU 확인 |
| 얼굴 변형 | IP-Adapter weight 조절 (0.7-0.85) |

---

## 💬 커뮤니티
- [Reddit r/comfyui](https://www.reddit.com/r/comfyui/)
- [Reddit r/StableDiffusion](https://www.reddit.com/r/StableDiffusion/)
- [ComfyUI Discord](https://discord.gg/comfyui)

---

<div align="center">

**Made with ❤️ for AI Artists**

</div>